In [2]:
import pandas as pd
import numpy as np

# ============================
# PARAMETERS
# ============================

SIMULATIONS = 1000
SHORTAGE_PENALTY = 10
HOLDING_PENALTY = 1
AVAILABLE_HOURS = 22

# ============================
# LOAD FILES
# ============================

stats = pd.read_excel("C:/Users/Ex0164/Book1.xlsx", sheet_name="Sheet2")
daily = pd.read_excel("C:/Users/Ex0164/Important codes/Child_for_26feb_actual.xlsx", sheet_name="Sheet1")

data = stats.merge(daily, on="Material")

# ============================
# PRODUCTION RATE
# ============================

data["Rate"] = (3600 / data["Cycle time "]) * data["cavity"]

# ============================
# OPTIMIZATION FUNCTION
# ============================

def optimize_part(row):

    inventory = row["Inventory on 24th"]
    demand_tomorrow = row["2026-02-27 Total Production Plan"]
    tentative_future = row["2026-03-01 Total Production Plan"]
    std = row["Std_Deviation"]
    rate = row["Rate"]

    best_qty = 0
    best_cost = np.inf

    max_qty = row["Feb INDENT"] * 1.2
    for qty in np.arange(0, max_qty, 50):

        future_stock = inventory + qty - demand_tomorrow

        simulated_demand = np.random.normal(
            tentative_future, std, SIMULATIONS
        )

        shortage = np.maximum(0, simulated_demand - future_stock)

        expected_shortage = shortage.mean()
        expected_inventory = max(0, future_stock - simulated_demand.mean())

        cost = (
            SHORTAGE_PENALTY * expected_shortage +
            HOLDING_PENALTY * expected_inventory
        )

        hours_needed = qty / rate

        if hours_needed <= AVAILABLE_HOURS and cost < best_cost:
            best_cost = cost
            best_qty = qty

    return best_qty, best_cost

# ============================
# RUN OPTIMIZATION
# ============================

results = data.apply(optimize_part, axis=1)

data["Planned_Qty"] = [r[0] for r in results]
data["Cost"] = [r[1] for r in results]

data.to_excel("daily_plan.xlsx", index=False)

print("✅ Optimization complete")

✅ Optimization complete


In [7]:
import pandas as pd
import numpy as np

# ============================
# PARAMETERS
# ============================

SIMULATIONS = 1000
SHORTAGE_PENALTY = 10
HOLDING_PENALTY = 1
AVAILABLE_HOURS = 22

SETUP_TIME = 40  # minutes
SETUP_HOURS = SETUP_TIME / 60

# ============================
# LOAD FILES
# ============================

stats = pd.read_excel("C:/Users/Ex0164/Book1.xlsx", sheet_name="Sheet2")
daily = pd.read_excel("C:/Users/Ex0164/Important codes/Child_for_28feb_actual.xlsx", sheet_name="Sheet1")

data = stats.merge(daily, on="Material")

# ============================
# PRODUCTION RATE
# ============================

data["Rate"] = (3600 / data["Cycle time "]) * data["cavity"]

# ============================
# OPTIMIZATION FUNCTION
# ============================

def optimize_part(row):

    inventory = row["Inventory on 24th"]
    demand_tomorrow = row["Sum of 28-02-2026"]
    tentative_future = row["Sum of 02-03-2026"]
    std = row["Std_Deviation"]
    rate = row["Rate"]

    best_qty = 0
    best_cost = np.inf

    max_qty = row["Feb INDENT"] * 1.2

    for qty in np.arange(0, max_qty + 1, 50):

        future_stock = inventory + qty - demand_tomorrow

        simulated_demand = np.random.normal(
            tentative_future, std, SIMULATIONS
        )

        shortage = np.maximum(0, simulated_demand - future_stock)

        expected_shortage = shortage.mean()
        expected_inventory = max(0, future_stock - simulated_demand.mean())

        # Setup penalty (only if production happens)
        setup_penalty = rate * SETUP_HOURS if qty > 0 else 0

        cost = (
            SHORTAGE_PENALTY * expected_shortage +
            HOLDING_PENALTY * expected_inventory +
            setup_penalty
        )

        hours_needed = qty / rate

        if hours_needed <= AVAILABLE_HOURS and cost < best_cost:
            best_cost = cost
            best_qty = qty

    return best_qty, best_cost

# ============================
# RUN OPTIMIZATION
# ============================

results = data.apply(optimize_part, axis=1)

data["Planned_Qty"] = [r[0] for r in results]
data["Cost"] = [r[1] for r in results]

# ============================
# TIME REQUIRED COLUMN
# ============================

data["Time Required (hrs)"] = data["Planned_Qty"] / data["Rate"]

# ============================
# SAVE OUTPUT
# ============================

data.to_excel("daily_plan.xlsx", index=False)

print("✅ Optimization complete")

✅ Optimization complete


In [4]:
import pandas as pd
import numpy as np

# ============================
# PARAMETERS
# ============================

SIMULATIONS = 1000
SHORTAGE_PENALTY = 10
HOLDING_PENALTY = 1
AVAILABLE_HOURS = 22

SETUP_TIME = 40  # minutes
SETUP_HOURS = SETUP_TIME / 60

# ============================
# LOAD FILES
# ============================

stats = pd.read_excel("C:/Users/Ex0164/Book1.xlsx", sheet_name="Sheet2")
daily = pd.read_excel("C:/Users/Ex0164/Important codes/Child_for_28feb_actual.xlsx", sheet_name="Sheet1")

data = stats.merge(daily, on="Material")

# ============================
# PRODUCTION RATE
# ============================

data["Rate"] = (3600 / data["Cycle time "]) * data["cavity"]

# ============================
# OPTIMIZATION FUNCTION
# ============================

def optimize_part(row):

    inventory = row["Inventory on 24th"]
    demand_tomorrow = row["Sum of 28-02-2026"]
    tentative_future = row["Sum of 02-03-2026"]
    std = row["Std_Deviation"]
    rate = row["Rate"]
    indent = row["Feb INDENT"]

    best_qty = 0
    best_cost = np.inf

    # Avoid overproduction
    target_stock = demand_tomorrow + tentative_future + std
    max_qty = max(0, target_stock - inventory)
    max_qty = min(max_qty, indent * 1.2)

    for qty in np.arange(0, max_qty + 1, 50):

        future_stock = inventory + qty - demand_tomorrow

        simulated_demand = np.maximum(
            0,
            np.random.normal(tentative_future, std, SIMULATIONS)
        )

        shortage = np.maximum(0, simulated_demand - future_stock)

        expected_shortage = shortage.mean()
        expected_inventory = max(0, future_stock - simulated_demand.mean())

        setup_penalty = rate * SETUP_HOURS if qty > 0 else 0

        cost = (
            SHORTAGE_PENALTY * expected_shortage +
            HOLDING_PENALTY * expected_inventory +
            setup_penalty
        )

        # Protect tomorrow
        tomorrow_shortage = max(0, demand_tomorrow - (inventory + qty))
        cost += SHORTAGE_PENALTY * tomorrow_shortage * 2

        # Setup consumes time
        hours_needed = (qty / rate + SETUP_HOURS) if qty > 0 else 0

        if hours_needed <= AVAILABLE_HOURS and cost < best_cost:
            best_cost = cost
            best_qty = qty

    # ================================
    # STRATEGIC CLOSE LOGIC
    # ================================

    full_time = indent / rate + SETUP_HOURS

    if full_time <= AVAILABLE_HOURS:

        remaining = max(0, indent - (inventory + best_qty))

        if tentative_future > 0:
            future_runs = remaining / tentative_future
        else:
            future_runs = 0

        setup_saving = future_runs * rate * SETUP_HOURS

        strategic_bonus = setup_saving * 0.3

        full_cost = HOLDING_PENALTY * max(0, indent - (inventory + demand_tomorrow))

        adjusted_full_cost = full_cost - strategic_bonus

        if adjusted_full_cost < best_cost:
            best_qty = indent
            best_cost = adjusted_full_cost

    return best_qty, best_cost

# ============================
# RUN OPTIMIZATION
# ============================

results = data.apply(optimize_part, axis=1)

data["Planned_Qty"] = [r[0] for r in results]
data["Cost"] = [r[1] for r in results]

# ============================
# TIME REQUIRED COLUMN
# ============================

data["Time Required (hrs)"] = np.where(
    data["Planned_Qty"] > 0,
    (data["Planned_Qty"] / data["Rate"]) + SETUP_HOURS,
    0
)

# ============================
# SAVE OUTPUT
# ============================

data.to_excel("daily_plan.xlsx", index=False)

print("✅ Optimization complete")

✅ Optimization complete


In [8]:
import pandas as pd
import numpy as np

# ============================
# PARAMETERS
# ============================

SIMULATIONS = 1000
SHORTAGE_PENALTY = 10
HOLDING_PENALTY = 1
AVAILABLE_HOURS = 22

SETUP_TIME = 40  # minutes
SETUP_HOURS = SETUP_TIME / 60

# ============================
# LOAD FILES
# ============================

stats = pd.read_excel("C:/Users/Ex0164/Book1.xlsx", sheet_name="Sheet2")
daily = pd.read_excel("C:/Users/Ex0164/Important codes/Child_for_17feb_actual.xlsx", sheet_name="Sheet1")

data = stats.merge(daily, on="Material")

# ============================
# PRODUCTION RATE
# ============================

data["Rate"] = (3600 / data["Cycle time "]) * data["cavity"]

# ============================
# OPTIMIZATION FUNCTION
# ============================

def optimize_part(row):

    inventory = row["Inventory on 24th"]
    demand_tomorrow = row["2026-02-18 Total Production Plan"]
    tentative_future = row["2026-02-20 Total Production Plan"]
    std = row["Std_Deviation"]
    rate = row["Rate"]

    best_qty = 0
    best_cost = np.inf

    max_qty = row["Feb INDENT"] * 1.2

    for qty in np.arange(0, max_qty + 1, 50):

        future_stock = inventory + qty - demand_tomorrow

        simulated_demand = np.random.normal(
            tentative_future, std, SIMULATIONS
        )

        shortage = np.maximum(0, simulated_demand - future_stock)

        expected_shortage = shortage.mean()
        expected_inventory = max(0, future_stock - simulated_demand.mean())

        # Setup penalty (only if production happens)
        setup_penalty = rate * SETUP_HOURS if qty > 0 else 0

        cost = (
            SHORTAGE_PENALTY * expected_shortage +
            HOLDING_PENALTY * expected_inventory +
            setup_penalty
        )

        hours_needed = qty / rate

        if hours_needed <= AVAILABLE_HOURS and cost < best_cost:
            best_cost = cost
            best_qty = qty

    return best_qty, best_cost

# ============================
# RUN OPTIMIZATION
# ============================

results = data.apply(optimize_part, axis=1)

data["Planned_Qty"] = [r[0] for r in results]
data["Cost"] = [r[1] for r in results]

# ============================
# TIME REQUIRED COLUMN
# ============================

data["Time Required (hrs)"] = data["Planned_Qty"] / data["Rate"]

# ============================
# SAVE OUTPUT
# ============================

data.to_excel("daily_plan_17feb.xlsx", index=False)

print("✅ Optimization complete")

✅ Optimization complete


In [14]:
import pandas as pd
import numpy as np

# ============================
# LOAD FILES
# ============================

plan = pd.read_excel("C:/Users/Ex0164/Important codes/daily_plan_17feb.xlsx")
prod = pd.read_excel("D:/farukhnagar plant part reports/Machine Part Report Plant 17 feb.xlsx")

# ============================
# SUMMARIZE PRODUCTION
# ============================

prod_summary = prod.groupby("Material").agg({
    "Ok": "sum",
    "Rej": "sum"
}).reset_index()

prod_summary.rename(columns={
    "Ok": "Produced_Qty",
    "Rej": "Rejected_Qty"
}, inplace=True)

# ============================
# OUTER MERGE (VERY IMPORTANT)
# ============================

data = plan.merge(prod_summary, on="Material", how="outer")

data.fillna(0, inplace=True)

# ============================
# EXECUTION STATUS
# ============================

data["Execution_Status"] = np.where(
    (data["Planned_Qty"] > 0) & (data["Produced_Qty"] == 0),
    "Planned but Not Produced",

    np.where(
        (data["Planned_Qty"] == 0) & (data["Produced_Qty"] > 0),
        "Produced but Not Planned",

        "Planned & Produced"
    )
)

# ============================
# PLAN vs ACTUAL
# ============================

data["Production_Gap"] = data["Produced_Qty"] - data["Planned_Qty"]

data["Plan_Status"] = np.where(
    data["Produced_Qty"] > data["Planned_Qty"],
    "Produced More",
    np.where(
        data["Produced_Qty"] < data["Planned_Qty"],
        "Produced Less",
        "Matched"
    )
)

# ============================
# INDENT COMPLETION
# ============================

data["Complete_Indent"] = data["Produced_Qty"] >= data["Feb INDENT"]

# ============================
# MACHINE-WISE PRODUCTION
# ============================

machine_summary = prod.groupby("Machine_Line")["Ok"].sum().reset_index()
machine_summary.rename(columns={"Ok": "Machine_Total_Production"}, inplace=True)

# ============================
# DASHBOARD DATASET
# ============================

dashboard = data[[
    "Material",
    "Planned_Qty",
    "Produced_Qty",
    "Feb INDENT",
    "Execution_Status",
    "Plan_Status",
    "Production_Gap",
    "Complete_Indent"
]]

# ============================
# SAVE EXCEL REPORT
# ============================

with pd.ExcelWriter("comparison_report_17feb.xlsx") as writer:
    data.to_excel(writer, sheet_name="Part_Comparison", index=False)
    machine_summary.to_excel(writer, sheet_name="Machine_Summary", index=False)

# ============================
# SAVE DASHBOARD FILE
# ============================

dashboard.to_excel("dashboard_data_17feb.xlsx", index=False)

print("✅ Excel Comparison Report Generated")
print("✅ Dashboard File Generated")

✅ Excel Comparison Report Generated
✅ Dashboard File Generated


In [ ]:
COLUMN NAMES = Material	Area	Mean_Actual	Std_Deviation	Lower_Actual_90	Upper_Actual_90	Inventory on 24th	Feb INDENT	Cycle time 	cavity	Line name	Segment	2026-02-18 Total Production Plan	2026-02-19 Total Production Plan	2026-02-20 Total Production Plan	Rate	Planned_Qty	Cost	Time Required (hrs)
COLUMN NAMES = Date	Machine_Line	Tonnage	Auto/Manual	SH	Material	Part Name	Ok	Rej	Prodn	RunTm Tgt	CycleTime Tgt	CycleTime Act	Planned Cavity	Actual Cavity	PE	Run Time	Pdt Time	CO Time	Downtime	Break Time	Cavity Part Loss	Cavity Eff Loss	Customer


In [20]:
import pandas as pd
import numpy as np

# ============================
# LOAD FILES
# ============================

plan = pd.read_excel("C:/Users/Ex0164/Important codes/daily_plan_17feb.xlsx")
prod = pd.read_excel("D:/farukhnagar plant part reports/Machine Part Report Plant 17 feb.xlsx")

# ============================
# SUMMARIZE PRODUCTION
# ============================

prod_summary = prod.groupby("Material").agg({
    "Ok": "sum",
    "Rej": "sum",
    "Machine_Line": "first"   # actual machine used
}).reset_index()

prod_summary.rename(columns={
    "Ok": "Produced_Qty",
    "Rej": "Rejected_Qty",
    "Machine_Line": "Actual_Machine"
}, inplace=True)

# ============================
# RENAME PLANNED MACHINE
# ============================

plan.rename(columns={"Line name": "Planned_Machine"}, inplace=True)

# ============================
# OUTER MERGE
# ============================

data = plan.merge(prod_summary, on="Material", how="outer")

data.fillna(0, inplace=True)

# ============================
# MACHINE MATCH CHECK
# ============================

data["Machine_Match"] = np.where(
    data["Planned_Machine"] == data["Actual_Machine"],
    "Same Machine",
    "Different Machine"
)

# ============================
# EXECUTION STATUS
# ============================

data["Execution_Status"] = np.where(
    (data["Planned_Qty"] > 0) & (data["Produced_Qty"] == 0),
    "Planned but Not Produced",

    np.where(
        (data["Planned_Qty"] == 0) & (data["Produced_Qty"] > 0),
        "Produced but Not Planned",

        "Planned & Produced"
    )
)

# ============================
# PLAN vs ACTUAL
# ============================

data["Production_Gap"] = data["Produced_Qty"] - data["Planned_Qty"]

data["Plan_Status"] = np.where(
    data["Produced_Qty"] > data["Planned_Qty"],
    "Produced More",
    np.where(
        data["Produced_Qty"] < data["Planned_Qty"],
        "Produced Less",
        "Matched"
    )
)

# ============================
# INDENT COMPLETION
# ============================

data["Complete_Indent"] = data["Produced_Qty"] >= data["Feb INDENT"]

# ============================
# MACHINE-WISE PRODUCTION
# ============================

machine_summary = prod.groupby("Machine_Line")["Ok"].sum().reset_index()
machine_summary.rename(columns={"Ok": "Machine_Total_Production"}, inplace=True)

# ============================
# DASHBOARD DATASET
# ============================

dashboard = data[[
    "Material",
    "Planned_Machine",
    "Actual_Machine",
    "Machine_Match",
    "Planned_Qty",
    "Produced_Qty",
    "Feb INDENT",
    "Execution_Status",
    "Plan_Status",
    "Production_Gap",
    "Complete_Indent"
]]

# ============================
# SAVE EXCEL REPORT
# ============================

with pd.ExcelWriter("comparison_report_17feb.xlsx") as writer:
    data.to_excel(writer, sheet_name="Part_Comparison", index=False)
    machine_summary.to_excel(writer, sheet_name="Machine_Summary", index=False)

# ============================
# SAVE DASHBOARD FILE
# ============================

dashboard.to_excel("dashboard_data_17feb.xlsx", index=False)

print("✅ Excel Comparison Report Generated")
print("✅ Dashboard File Generated")

✅ Excel Comparison Report Generated
✅ Dashboard File Generated


In [18]:
import pandas as pd
import numpy as np

# ============================
# PARAMETERS
# ============================

SIMULATIONS = 1000
SHORTAGE_PENALTY = 10
HOLDING_PENALTY = 1
AVAILABLE_HOURS = 22

SETUP_TIME = 40  # minutes
SETUP_HOURS = SETUP_TIME / 60

# ============================
# LOAD FILES
# ============================

stats = pd.read_excel("C:/Users/Ex0164/Book1.xlsx", sheet_name="Sheet2")
daily = pd.read_excel("C:/Users/Ex0164/Important codes/Child_for_17feb_actual.xlsx", sheet_name="Sheet1")

data = stats.merge(daily, on="Material")

# ============================
# PRODUCTION RATE
# ============================

data["Rate"] = (3600 / data["Cycle time "]) * data["cavity"]

# ============================
# MONTE CARLO OPTIMIZATION
# ============================

def optimize_part(row):

    inventory = row["Inventory on 24th"]
    demand_tomorrow = row["2026-02-18 Total Production Plan"]
    tentative_future = row["2026-02-20 Total Production Plan"]
    std = row["Std_Deviation"]
    rate = row["Rate"]

    best_qty = 0
    best_cost = np.inf

    max_qty = row["Feb INDENT"] * 1.2

    for qty in np.arange(0, max_qty + 1, 50):

        future_stock = inventory + qty - demand_tomorrow

        simulated_demand = np.random.normal(
            tentative_future, std, SIMULATIONS
        )

        shortage = np.maximum(0, simulated_demand - future_stock)

        expected_shortage = shortage.mean()
        expected_inventory = max(0, future_stock - simulated_demand.mean())

        setup_penalty = rate * SETUP_HOURS if qty > 0 else 0

        cost = (
            SHORTAGE_PENALTY * expected_shortage +
            HOLDING_PENALTY * expected_inventory +
            setup_penalty
        )

        hours_needed = qty / rate

        if hours_needed <= AVAILABLE_HOURS and cost < best_cost:
            best_cost = cost
            best_qty = qty

    return best_qty, best_cost

# ============================
# RUN OPTIMIZATION
# ============================

results = data.apply(optimize_part, axis=1)

data["Planned_Qty"] = [r[0] for r in results]
data["Cost"] = [r[1] for r in results]

# ============================
# HOURS REQUIRED PER PART
# ============================

data["Hours_Required"] = data["Planned_Qty"] / data["Rate"]

# ============================
# MACHINE CAPACITY CHECK
# ============================

machine_load = data.groupby("Line name")["Hours_Required"].sum().reset_index()
machine_load.rename(columns={"Hours_Required": "Total_Hours"}, inplace=True)

# ============================
# ADJUST IF MACHINE OVERLOAD
# ============================

for machine in machine_load["Line name"]:

    total_hours = machine_load.loc[
        machine_load["Line name"] == machine,
        "Total_Hours"
    ].values[0]

    if total_hours > AVAILABLE_HOURS:

        scaling_factor = AVAILABLE_HOURS / total_hours

        mask = data["Line name"] == machine

        data.loc[mask, "Planned_Qty"] = (
            data.loc[mask, "Planned_Qty"] * scaling_factor
        )

        data.loc[mask, "Planned_Qty"] = (
            data.loc[mask, "Planned_Qty"].round(-1)
        )

# ============================
# FINAL HOURS CALCULATION
# ============================

data["Time Required (hrs)"] = data["Planned_Qty"] / data["Rate"]

# ============================
# SAVE OUTPUT
# ============================

data.to_excel("daily_plan_17feb.xlsx", index=False)

print("✅ Machine-Aware Optimization Complete")

✅ Machine-Aware Optimization Complete


In [22]:
import pandas as pd
import numpy as np

SIMULATIONS = 1000
SHORTAGE_PENALTY = 10
HOLDING_PENALTY = 1
AVAILABLE_HOURS = 22

SETUP_TIME = 40
SETUP_HOURS = SETUP_TIME / 60

MIN_RUN_HOURS = 4
MAX_PARTS_PER_MACHINE = 3

stats = pd.read_excel("C:/Users/Ex0164/Book1.xlsx", sheet_name="Sheet2")
daily = pd.read_excel("C:/Users/Ex0164/Important codes/Child_for_17feb_actual.xlsx", sheet_name="Sheet1")

data = stats.merge(daily, on="Material")

data["Rate"] = (3600 / data["Cycle time "]) * data["cavity"]

# ============================
# PRIMARY OPTIMIZATION (NO INVENTORY)
# ============================

def optimize_no_inventory(row):

    inventory = 0
    demand_tomorrow = row["2026-02-18 Total Production Plan"]
    tentative_future = row["2026-02-20 Total Production Plan"]
    std = row["Std_Deviation"]
    rate = row["Rate"]

    best_qty = 0
    best_cost = np.inf
    max_qty = row["Feb INDENT"] * 1.2

    for qty in np.arange(0, max_qty + 1, 50):

        future_stock = qty - demand_tomorrow

        simulated_demand = np.random.normal(
            tentative_future, std, SIMULATIONS
        )

        shortage = np.maximum(0, simulated_demand - future_stock)

        expected_shortage = shortage.mean()
        expected_inventory = max(0, future_stock - simulated_demand.mean())

        setup_penalty = rate * SETUP_HOURS if qty > 0 else 0

        cost = (
            SHORTAGE_PENALTY * expected_shortage +
            HOLDING_PENALTY * expected_inventory +
            setup_penalty
        )

        hours_needed = qty / rate

        if hours_needed < MIN_RUN_HOURS and qty > 0:
            continue

        if hours_needed <= AVAILABLE_HOURS and cost < best_cost:
            best_cost = cost
            best_qty = qty

    return best_qty

data["Planned_Qty"] = data.apply(optimize_no_inventory, axis=1)

data["Hours_Required"] = data["Planned_Qty"] / data["Rate"]

# ============================
# CHANGEOVER MINIMIZATION
# ============================

data = data.sort_values(by=["Line name", "Hours_Required"], ascending=False)
data["Rank"] = data.groupby("Line name")["Hours_Required"].rank(method="first", ascending=False)
data.loc[data["Rank"] > MAX_PARTS_PER_MACHINE, "Planned_Qty"] = 0

data["Hours_Required"] = data["Planned_Qty"] / data["Rate"]

# ============================
# MACHINE UTILIZATION CHECK
# ============================

machine_load = data.groupby("Line name")["Hours_Required"].sum().reset_index()

# ============================
# INVENTORY AS FALLBACK
# ============================

for machine in machine_load["Line name"]:

    total_hours = machine_load.loc[
        machine_load["Line name"] == machine,
        "Hours_Required"
    ].values[0]

    if total_hours < AVAILABLE_HOURS:

        idle_time = AVAILABLE_HOURS - total_hours

        mask = data["Line name"] == machine

        for idx in data[mask].index:

            row = data.loc[idx]

            inv = row["Inventory on 24th"]
            rate = row["Rate"]

            if inv > 0:

                possible_hours = inv / rate

                add_hours = min(possible_hours, idle_time)

                add_qty = add_hours * rate

                data.loc[idx, "Planned_Qty"] += add_qty

                idle_time -= add_hours

            if idle_time <= 0:
                break

data["Planned_Qty"] = data["Planned_Qty"].round(-1)

data["Time Required (hrs)"] = data["Planned_Qty"] / data["Rate"]

data.to_excel("daily_plan_17feb.xlsx", index=False)

print("✅ Smart Inventory-Aware Machine Planning Complete")

✅ Smart Inventory-Aware Machine Planning Complete


In [23]:
import pandas as pd
import numpy as np

# ============================
# LOAD FILES
# ============================

plan = pd.read_excel("C:/Users/Ex0164/Important codes/daily_plan_17feb.xlsx")
prod = pd.read_excel("D:/farukhnagar plant part reports/Machine Part Report Plant 17 feb.xlsx")

# ============================
# SUMMARIZE PRODUCTION
# ============================

prod_summary = prod.groupby("Material").agg({
    "Ok": "sum",
    "Rej": "sum",
    "Machine_Line": "first"   # actual machine used
}).reset_index()

prod_summary.rename(columns={
    "Ok": "Produced_Qty",
    "Rej": "Rejected_Qty",
    "Machine_Line": "Actual_Machine"
}, inplace=True)

# ============================
# RENAME PLANNED MACHINE
# ============================

plan.rename(columns={"Line name": "Planned_Machine"}, inplace=True)

# ============================
# OUTER MERGE
# ============================

data = plan.merge(prod_summary, on="Material", how="outer")

data.fillna(0, inplace=True)

# ============================
# MACHINE MATCH CHECK
# ============================

data["Machine_Match"] = np.where(
    data["Planned_Machine"] == data["Actual_Machine"],
    "Same Machine",
    "Different Machine"
)

# ============================
# EXECUTION STATUS
# ============================

data["Execution_Status"] = np.where(
    (data["Planned_Qty"] > 0) & (data["Produced_Qty"] == 0),
    "Planned but Not Produced",

    np.where(
        (data["Planned_Qty"] == 0) & (data["Produced_Qty"] > 0),
        "Produced but Not Planned",

        "Planned & Produced"
    )
)

# ============================
# PLAN vs ACTUAL
# ============================

data["Production_Gap"] = data["Produced_Qty"] - data["Planned_Qty"]

data["Plan_Status"] = np.where(
    data["Produced_Qty"] > data["Planned_Qty"],
    "Produced More",
    np.where(
        data["Produced_Qty"] < data["Planned_Qty"],
        "Produced Less",
        "Matched"
    )
)

# ============================
# INDENT COMPLETION
# ============================

data["Complete_Indent"] = data["Produced_Qty"] >= data["Feb INDENT"]

# ============================
# MACHINE-WISE PRODUCTION
# ============================

machine_summary = prod.groupby("Machine_Line")["Ok"].sum().reset_index()
machine_summary.rename(columns={"Ok": "Machine_Total_Production"}, inplace=True)

# ============================
# DASHBOARD DATASET
# ============================

dashboard = data[[
    "Material",
    "Planned_Machine",
    "Actual_Machine",
    "Machine_Match",
    "Planned_Qty",
    "Produced_Qty",
    "Feb INDENT",
    "Execution_Status",
    "Plan_Status",
    "Production_Gap",
    "Complete_Indent"
]]

# ============================
# SAVE EXCEL REPORT
# ============================

with pd.ExcelWriter("comparison_report_17feb.xlsx") as writer:
    data.to_excel(writer, sheet_name="Part_Comparison", index=False)
    machine_summary.to_excel(writer, sheet_name="Machine_Summary", index=False)

# ============================
# SAVE DASHBOARD FILE
# ============================

dashboard.to_excel("dashboard_data_17feb.xlsx", index=False)

print("✅ Excel Comparison Report Generated")
print("✅ Dashboard File Generated")

✅ Excel Comparison Report Generated
✅ Dashboard File Generated


In [1]:
import pandas as pd
import numpy as np

# ============================
# LOAD FILES
# ============================

plan = pd.read_excel("C:/Users/Ex0164/Important codes/daily_plan_17feb.xlsx")
prod = pd.read_excel("D:/farukhnagar plant part reports/Machine Part Report Plant 17 feb.xlsx")

# ============================
# SUMMARIZE PRODUCTION
# ============================

prod_summary = prod.groupby("Material").agg({
    "Ok": "sum",
    "Rej": "sum",
    "Machine_Line": "first"
}).reset_index()

prod_summary.rename(columns={
    "Ok": "Produced_Qty",
    "Rej": "Rejected_Qty",
    "Machine_Line": "Actual_Machine"
}, inplace=True)

# ============================
# RENAME PLANNED MACHINE
# ============================

plan.rename(columns={"Line name": "Planned_Machine"}, inplace=True)

# ============================
# MERGE DATA
# ============================

data = plan.merge(prod_summary, on="Material", how="outer")
data.fillna(0, inplace=True)

# ============================
# RATE & TIME CALCULATION
# ============================

data["Rate"] = (3600 / data["Cycle time "]) * data["cavity"]
data["Actual_Time_Required"] = data["Produced_Qty"] / data["Rate"]

# ============================
# MACHINE MATCH CHECK
# ============================

data["Machine_Match"] = np.where(
    data["Planned_Machine"] == data["Actual_Machine"],
    "Same Machine",
    "Different Machine"
)

# ============================
# EXECUTION STATUS
# ============================

data["Execution_Status"] = np.where(
    (data["Planned_Qty"] > 0) & (data["Produced_Qty"] == 0),
    "Planned but Not Produced",

    np.where(
        (data["Planned_Qty"] == 0) & (data["Produced_Qty"] > 0),
        "Produced but Not Planned",

        "Planned & Produced"
    )
)

# ============================
# PLAN vs ACTUAL
# ============================

data["Production_Gap"] = data["Produced_Qty"] - data["Planned_Qty"]

data["Plan_Status"] = np.where(
    data["Produced_Qty"] > data["Planned_Qty"],
    "Produced More",
    np.where(
        data["Produced_Qty"] < data["Planned_Qty"],
        "Produced Less",
        "Matched"
    )
)

# ============================
# INDENT COMPLETION
# ============================

data["Complete_Indent"] = data["Produced_Qty"] >= data["Feb INDENT"]

# ============================
# MACHINE PART DETAIL
# ============================

machine_part_detail = data[[
    "Actual_Machine",
    "Material",
    "Produced_Qty",
    "Actual_Time_Required"
]]

machine_part_detail = machine_part_detail[machine_part_detail["Produced_Qty"] > 0]

# ============================
# MACHINE SUMMARY
# ============================

machine_summary = machine_part_detail.groupby("Actual_Machine").agg({
    "Material": lambda x: ", ".join(x.astype(str)),
    "Produced_Qty": "sum",
    "Actual_Time_Required": "sum"
}).reset_index()

machine_summary.rename(columns={
    "Material": "Parts_Made",
    "Produced_Qty": "Total_Qty",
    "Actual_Time_Required": "Total_Hours_Used"
}, inplace=True)

# ============================
# DASHBOARD DATASET
# ============================

dashboard = data[[
    "Material",
    "Planned_Machine",
    "Actual_Machine",
    "Machine_Match",
    "Planned_Qty",
    "Produced_Qty",
    "Feb INDENT",
    "Execution_Status",
    "Plan_Status",
    "Production_Gap",
    "Complete_Indent"
]]

# ============================
# SAVE OUTPUT
# ============================

with pd.ExcelWriter("comparison_report_17feb.xlsx") as writer:
    data.to_excel(writer, sheet_name="Part_Comparison", index=False)
    machine_summary.to_excel(writer, sheet_name="Machine_Summary", index=False)
    machine_part_detail.to_excel(writer, sheet_name="Machine_Part_Detail", index=False)

dashboard.to_excel("dashboard_data_17feb.xlsx", index=False)

print("✅ Excel Comparison Report Generated")
print("✅ Dashboard File Generated")

✅ Excel Comparison Report Generated
✅ Dashboard File Generated
